# Setup Environment

### Only Run on First Set-Up

In [1]:

%%bash
set -eo pipefail
ENV_NAME="rag-capstone" PLATFORM="linux-64" bash scripts/bootstrap_env.sh


mkdir -p "$CONDA_PREFIX/etc/conda/activate.d"
cat > "$CONDA_PREFIX/etc/conda/activate.d/java.sh" <<'EOF'

set -euo pipefail

export JAVA_HOME="${CONDA_PREFIX}"
export PATH="${CONDA_PREFIX}/bin:${PATH}"
EOF
chmod +x "$CONDA_PREFIX/etc/conda/activate.d/java.sh"


== Bootstrapping conda env ==
ENV_NAME:     rag-capstone
TOOLING_ENV:  tooling
LOCK_FILE:    conda-lock.yml
PLATFORM:     linux-64
ENV_YML:      environment.yml
PIP_REQS:     requirements-pip.txt

== Tooling versions ==
conda-lock, version 4.0.0
2.5.0

== Generating lockfile ==


Locking dependencies for ['linux-64']...
INFO:conda_lock.conda_solver:linux-64 using specs ['python 3.11.*', 'pip', 'openjdk 21.*', 'numpy 2.0.2.*', 'pandas 2.2.3.*', 'streamlit 1.54.0.*', 'pyarrow', 'pytorch', 'cpuonly', 'ipykernel']
 - Install lock using: conda-lock install --name YOURENV conda-lock.yml


== Recreating env: rag-capstone ==
== Installing pip requirements ==
  Using cached pyserini-1.5.0-py3-none-any.whl
  Using cached faiss_cpu-1.13.2-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (7.6 kB)
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Using cached sentence_transformers-5.2.3-py3-none-any.whl.metadata (16 kB)
  Using cached datasets-4.0.0-py3-none-any.whl.metadata (19 kB)
  Using cached ragas-0.4.3-py3-none-any.whl.metadata (23 kB)
  Using cached litellm-1.82.0-py3-none-any.whl.metadata (30 kB)
  Using cached mlflow-3.10.1-py3-none-any.whl.metadata (31 kB)
  Using cached sagemaker_mlflow-0.2.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached langgraph-1.0.10-py3-none-any.whl.metadata (7.4 kB)
  Using cached langchain_core-1.2.16-py3-none-any.whl.metadata (4.4 kB)
  Using cached boto3-1.42.60-py3-none-any.whl.metadata (6.7 kB)
  Using cached botocore-1.42.60-py3-none-any.whl.metadata (5.9 kB)
  Using cached tqdm-4.67.3-py3-non

### Run each kernal restart

In [2]:
%%bash
source "$(conda info --base)/etc/profile.d/conda.sh"
conda activate rag-capstone

In [3]:
import os
import subprocess
from pathlib import Path

def _first_match(root: Path, name: str) -> str:
    for p in root.rglob(name):
        if p.is_file():
            return str(p)
    raise RuntimeError(f"Could not find {name} under {root}")

env_prefix = str(Path(os.sys.executable).resolve().parent.parent)

javac = _first_match(Path(env_prefix), "javac")
libjvm = _first_match(Path(env_prefix), "libjvm.so")

java_home = str(Path(libjvm).resolve().parent.parent.parent)

os.environ["CONDA_PREFIX"] = env_prefix
os.environ["JAVA_HOME"] = java_home
os.environ["JVM_PATH"] = libjvm
os.environ["PATH"] = f"{env_prefix}/bin:{java_home}/bin:" + os.environ.get("PATH", "")

print("Python:", os.sys.executable)
print("CONDA_PREFIX:", os.environ["CONDA_PREFIX"])
print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("JVM_PATH:", os.environ["JVM_PATH"])
print("javac:", subprocess.check_output(["which", "javac"]).decode().strip())
print(subprocess.check_output(["javac", "-version"], stderr=subprocess.STDOUT).decode().strip())

Python: /home/sagemaker-user/.conda/envs/rag-capstone/bin/python
CONDA_PREFIX: /home/sagemaker-user/.conda/envs/rag-capstone
JAVA_HOME: /home/sagemaker-user/.conda/envs/rag-capstone/lib/jvm
JVM_PATH: /home/sagemaker-user/.conda/envs/rag-capstone/lib/jvm/lib/server/libjvm.so
javac: /home/sagemaker-user/.conda/envs/rag-capstone/bin/javac
javac 21.0.10-internal


In [17]:
%%capture
from src.utils.aws_secrets import bootstrap_env
bootstrap_env()